# 99 — Teardown

Not part of the proof. Run it once the proof has been kept.

What it does, in order:

1. Prints the tables and views, so there is a record of what the demo owned.
2. Drops every DMF association. **This is the step that stops the billing.** Scheduled evaluations run on Snowflake-managed compute and bill as Data Quality Monitoring until the association is gone. Suspending the warehouse does not stop them.
3. Drops the views, the tables, the custom DMF, and the schema.
4. Suspends `BT_WH` last, as housekeeping, because that warehouse may be shared.

What it deliberately does not do: it does not drop the database, and it does not drop the warehouse. Never add either statement.

In [ ]:
USE ROLE BT_BUILDER_ROLE;
USE WAREHOUSE BT_WH;
USE DATABASE BT_DQ_REF;
USE SCHEMA BT_DQ_REF;

In [ ]:
SHOW TABLES;
SHOW VIEWS;

## Drop the associations

Each `DROP DATA METRIC FUNCTION` names the function and the same columns it was attached on. The custom DMF is dropped here too.

In [ ]:
ALTER TABLE IF EXISTS BT_DQ_REF.BT_DQ_REF.bt_orders
  DROP DATA METRIC FUNCTION SNOWFLAKE.CORE.DUPLICATE_COUNT ON (order_id, line_no);
ALTER TABLE IF EXISTS BT_DQ_REF.BT_DQ_REF.bt_orders
  DROP DATA METRIC FUNCTION BT_DQ_REF.BT_DQ_REF.bt_positive_qty ON (qty);
ALTER VIEW IF EXISTS BT_DQ_REF.BT_DQ_REF.bt_orders_latest_batch
  DROP DATA METRIC FUNCTION SNOWFLAKE.CORE.ROW_COUNT ON ();
ALTER VIEW IF EXISTS BT_DQ_REF.BT_DQ_REF.bt_orders_latest_freshness
  DROP DATA METRIC FUNCTION SNOWFLAKE.CORE.FRESHNESS ON (load_ts);

## Drop the objects

The function goes after its association. A custom DMF cannot be dropped while it is still attached to a table or a view.

In [ ]:
DROP VIEW IF EXISTS BT_DQ_REF.BT_DQ_REF.bt_orders_latest_freshness;
DROP VIEW IF EXISTS BT_DQ_REF.BT_DQ_REF.bt_orders_latest_batch;
DROP TABLE IF EXISTS BT_DQ_REF.BT_DQ_REF.bt_orders;
DROP TABLE IF EXISTS BT_DQ_REF.BT_DQ_REF.bt_load_ledger;
DROP TABLE IF EXISTS BT_DQ_REF.BT_DQ_REF.bt_batch_id;
DROP FUNCTION IF EXISTS BT_DQ_REF.BT_DQ_REF.bt_positive_qty(TABLE(NUMBER));

## Schema, then the shared warehouse

`BT_BUILDER_ROLE` owns the schema, so `DROP SCHEMA` succeeds. The database stays.

The suspend is last so the drops above still have a running warehouse, and it is housekeeping only. It is not a cost control. The association drops above are what stop the schedules.

In [ ]:
DROP SCHEMA IF EXISTS BT_DQ_REF.BT_DQ_REF;

In [ ]:
ALTER WAREHOUSE BT_WH SUSPEND;